## Self-attention
The self-attention is a mechanism to make model to understand the interaction/relationship between tokens.
In causal (autoregressive, in our case) models like GPT, each token can only attend (interact) to itself and previous tokens in the context.

### Mathmatic Trick
We use mean as a rudimentary implementaion to let each token embedding contains previous information within context.

### Simple sample
- bow: bag of words
- 3 implementation illustration
    - for loop: slow
    - matrix multiplication: fast
    - softmax
        - The probability distribution represents the affinities between token
        - The `-inf` means the future token can not communicate / has no affinities with the past tokens
#### torch.tril
```python
block_size = 4
mask = torch.tril(torch.ones(block_size, block_size))
# mask tensor:
# [[1., 0., 0., 0.],
#  [1., 1., 0., 0.],
#  [1., 1., 1., 0.],
#  [1., 1., 1., 1.]]
```


### Softmax Explanation

This code implements **causal (masked) attention** — ensuring each token can only attend to **past and present tokens**, not future ones.


### Line by Line

**1. Create a lower triangular mask**
```python
tril = torch.tril(torch.ones(T, T))
```
Creates a matrix that is `1` below and on the diagonal, `0` above:
```
1 0 0 0
1 1 0 0
1 1 1 0
1 1 1 1
```
The `0`s represent **future tokens** — positions a token should NOT see.


**2. Initialize weights as zero**
```python
wei = torch.zeros((T,T))
```
Starts with uniform zero affinities — every token pair has equal (zero) weight for now. The comment hints this is just a starting point; in real attention, these will be computed from **Q·K**.


**3. Mask out future tokens with `-inf`**
```python
wei = wei.masked_fill(tril == 0, float('-inf'))
```
Wherever `tril == 0` (i.e. future positions), replace with `-inf`:
```
0   -inf -inf -inf
0    0   -inf -inf
0    0    0   -inf
0    0    0    0
```
Why `-inf`? Because the next step is softmax, and `softmax(-inf) → 0`, effectively **erasing** those positions.

**4. Apply softmax to get attention weights**
```python
wei = F.softmax(wei, dim=-1)
```
Softmax converts each row into probabilities that sum to 1:
```
1.00  0.00  0.00  0.00   ← token 1 only sees itself
0.50  0.50  0.00  0.00   ← token 2 sees tokens 1-2
0.33  0.33  0.33  0.00   ← token 3 sees tokens 1-3
0.25  0.25  0.25  0.25   ← token 4 sees all tokens
```
The `-inf` values become exactly `0` after softmax — those future tokens contribute **nothing**.


**5. Weighted sum of values**
```python
xbow3 = wei @ x
```
Matrix multiply the attention weights with `x` (the value vectors). Each output token is now a **weighted average** of past tokens' information — exactly the "sum of values weighted by attention" the blog described!



**6. Sanity check**
```python
torch.allclose(xbow3, xbow2)
```
Verifies this softmax approach produces the same result as a previous implementation (`xbow2`), just written differently.


### Big Picture

This is the **masking** step in the Q/K/V attention we discussed — it enforces that the model is **causal** (can't cheat by looking at future tokens). In real attention, `wei` wouldn't start as zeros but would be computed as `Q · Kᵀ / √d` before the masking is applied.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
# (B,T,C)
# B: batch, T: timestep C: context size
B, T, C = 4, 8, 2
x = torch.randn(4,8,2)
x.shape

torch.Size([4, 8, 2])

In [93]:
# Solution 1 for loop
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1] # (t, C)
        xbow[b, t] = torch.mean(xprev, dim = 0)
xbow

tensor([[[ 1.8077e-01, -6.9988e-02, -3.5962e-01,  ..., -8.0164e-01,
           1.5236e+00,  2.5086e+00],
         [-2.4116e-01, -1.6063e-01,  3.2526e-01,  ...,  3.6581e-01,
           1.5667e+00,  1.0527e+00],
         [-4.3893e-01,  9.2179e-02,  1.9971e-01,  ...,  9.8210e-02,
           7.1071e-01,  5.6531e-01],
         ...,
         [-9.8921e-01,  1.3417e-01,  2.8014e-01,  ...,  3.4950e-01,
           6.1414e-01,  1.2510e-01],
         [-8.2062e-01,  6.6077e-02,  4.9662e-01,  ...,  3.5242e-01,
           4.4703e-01,  5.0332e-02],
         [-7.9077e-01,  3.0213e-02,  4.3624e-01,  ...,  6.9874e-02,
           3.2521e-01,  1.7912e-01]],

        [[ 4.5618e-01, -1.0917e+00, -8.2073e-01,  ...,  5.1187e-02,
          -6.5764e-01, -2.5729e+00],
         [ 2.3859e-01, -4.2831e-02, -1.0349e+00,  ...,  4.1852e-01,
          -9.0388e-01, -6.2984e-01],
         [ 8.9262e-01, -1.0170e-01, -5.0905e-01,  ...,  6.4184e-02,
          -2.4146e-01, -6.8638e-01],
         ...,
         [ 5.6778e-01,  3

In [94]:
# Solution 2 # matrix multiplication
wei = torch.tril(torch.ones(T, T), diagonal = 0)
wei = wei / torch.sum(wei, dim = 1, keepdim = True)
xbow2 = torch.zeros((B,T,C))
# (T, T) @ (B, T, C) -> (B, T, T) @ (B, T, C) = (B, T, C)
xbow2 = wei @ x
xbow2

tensor([[[ 1.8077e-01, -6.9988e-02, -3.5962e-01,  ..., -8.0164e-01,
           1.5236e+00,  2.5086e+00],
         [-2.4116e-01, -1.6063e-01,  3.2526e-01,  ...,  3.6581e-01,
           1.5667e+00,  1.0527e+00],
         [-4.3893e-01,  9.2179e-02,  1.9971e-01,  ...,  9.8210e-02,
           7.1071e-01,  5.6531e-01],
         ...,
         [-9.8921e-01,  1.3417e-01,  2.8014e-01,  ...,  3.4950e-01,
           6.1414e-01,  1.2510e-01],
         [-8.2062e-01,  6.6077e-02,  4.9662e-01,  ...,  3.5242e-01,
           4.4703e-01,  5.0332e-02],
         [-7.9077e-01,  3.0213e-02,  4.3624e-01,  ...,  6.9874e-02,
           3.2521e-01,  1.7912e-01]],

        [[ 4.5618e-01, -1.0917e+00, -8.2073e-01,  ...,  5.1187e-02,
          -6.5764e-01, -2.5729e+00],
         [ 2.3859e-01, -4.2831e-02, -1.0349e+00,  ...,  4.1852e-01,
          -9.0388e-01, -6.2984e-01],
         [ 8.9262e-01, -1.0170e-01, -5.0905e-01,  ...,  6.4184e-02,
          -2.4146e-01, -6.8638e-01],
         ...,
         [ 5.6778e-01,  3

In [95]:
torch.allclose(xbow, xbow2)

True

In [ ]:
# Solution 3 softmax
tril = torch.tril(torch.ones(T, T))
# In future, most of the affinities between tokens won't just be 0 (uniform distributed)
wei = torch.zeros((T,T))
# The future tokens have no affiliations with past tokens
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim = -1)
# (T, T) @ (B, T, C) -> (B, T, T) @ (B, T, C) = (B, T, C)
xbow3 = wei @ x

torch.allclose(xbow3, xbow2)

True

## Single Head self-attention
- Every token emmit 2 vectors
    - query: What I am looking for
    - key: What do I have/contain
- Weights = Attentions = Affinities bettwen tokens = dot product of query and key vectors
    - The entries of weight matrix means the affilitation between query of a token and a key of a token
    - $wei_{i,j} =Q_i \cdot K_j​$
        - i: query time step
        - j: key time step
$$
\text{wei} =
\begin{bmatrix}
Q_1K_1 & Q_1K_2 & \dots & Q_1K_8 \\
Q_2K_1 & Q_2K_2 & \dots & Q_2K_8 \\
\vdots & \vdots & \ddots & \vdots \\
Q_8K_1 & Q_8K_2 & \dots & Q_8K_8
\end{bmatrix}
$$

If the largest value in row i occurs at column j,
it means the i-th token is attending most strongly to the j-th token.
In other words, the representation at position i
considers the representation at position j most relevant
when computing its contextualized output.


attends meansd
    - The representation at position i uses information from position j when computing its new representation.
    - The representation at position i finds the representation at position j most relevant.


### Token
x is a batch of 4 examples (sequences), and each example has 8 tokens, and each token is represented by a 32-dim vector.


## Independent information channels
 We use V instead of X because attention separates:

- Q: what a token is looking for
- K: what a token contains for matching
- V: what information the token shares
### V
Using V allows:

- Dimensionality reduction (head_size < embedding_dim)
- Multi-head specialization
- Independent information channels

### Q and K
- Q·K decides relevance: Q and K determine attention weights (who attends to whom).
- V carries content: V determines the information that gets aggregated.

## Note
- x: embedding matrix, tokens of all sequences have been converted to embedding vectors
    - 4 sequences
    - Each sequence has 8 timesteps
    - Each token is a 32-dimensional embedding vector 

In [102]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

# Current token won't communicate with future tokens
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
# Probability distribution
wei = F.softmax(wei, dim=-1)

v = value(x) # (B,T,head_size)
out = wei @ v # (B,T,T) @ (B,T,head_size)
wei.shape

torch.Size([4, 8, 8])

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
     - In our case, we have 8 nodes/tokens for each batch. The first node points to itself, second node points to itself and first node..... etc.   
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
    - self-attention: Same source `x` produces both key and query
    - cross-attention: key and query come from different sets and we are finding relationships between two sequences.
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

## Types of Transformer

### Encoder
- No masking, Can see all tokens
- Full bidirectional attention
- Used when entire input is known
- Good for classification, understanding
- Takes an input sequence
- Processes the entire sequence
- Produces contextual representations
### Decoder
- Generates output autoregressively 
- Produces tokens one by one
- Has triangular (causal) masking
- Token can only see previous tokens
- Used for autoregressive generation
- Application: GPT, Gemini which are autoregressive using decoder to generate sequecnes one token at a time

### Overview 
Our self-attention code block is decoder.
Remove `wei = wei.masked_fill(tril == 0, float('-inf'))` then you get encoder block.

## Head
- A head is 1 independent Q/K/V projection / attention calculation / relationship
- Multi-head attention is: Several heads run in parallel
- attention is `out`

$$
Attention(Q, K, V ) = softmax(\frac{QK^T}{\sqrt{d_k}})V
$$
- $d_k$ is head size
- [Great Explanation](https://chatgpt.com/s/t_69aa3a37b8c0819180f335296f9d8ac6)

## Unit variance
- Z-score normalization: $\frac{X - \mu}{\sigma}$. Once the X has been thourhg z-score normalziaiton, the data distribution will have unit variance.
- Unit variance means the expected variacne is 1.
- Simple example [10, 20, 30]

### Why $\frac{1}{\sqrt{d_k}}$
- The $\frac{1}{\sqrt{d_k}}$ kept wei having unit variance
    - https://share.google/aimode/qCVQGBq3lT0geRxwR
- Influence softmax result
    - If Variance is Large (Not Unit Variance): The softmax distribution becomes "Saturated" or Sharp.
        - The Softmax function gives almost all the probability to the single largest value, while everything else becomes essentially zero.
        - The Danger: This leads to "Vanishing Gradients" because the slope of the Softmax curve becomes very flat at the extremes. The model stops learning.
    - If Variance is Unit (Scaled by ): The distribution is "Smooth."
        - The Softmax can spread probability more meaningfully across different keys.
        - The Benefit: This keeps the values in the "sweet spot" of the Softmax function where the gradients are strongest, allowing for stable training. 

In [119]:
k = torch.randn(B,T, head_size)
q = torch.randn(B,T, head_size)
wei = q @ k.transpose(-2, -1)
wei.var(), q.var(), k.var()

(tensor(14.6499), tensor(0.9473), tensor(0.9279))

In [120]:
wei = wei * head_size**(-0.5)
wei.var()

tensor(0.9156)

In [122]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim = -1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [126]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]) * 8, dim = -1)

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])